In [ ]:
import pandas as pd
import numpy as np
import os
from cva import *
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df_plain = pd.read_csv("data/black_cox_cva_no_control_timing_results.csv")
df_control = pd.read_csv("data/black_cox_cva_control_timing_results.csv")

print(df_control.shape)
print(df_plain.shape)

In [ ]:
parameter_cols = [
    "L_over_V0",
    "S0_over_K",
    "sigma_s",
    "sigma_v",
    "mu_v",
    "rho",
]

df = df_plain.merge(
    df_control,
    on=parameter_cols,
    suffixes=("_plain", "_control")
)

print(df.shape)

In [ ]:
df['var_reduction_factor'] = (df['standard_error_plain']/df['standard_error_control'])**2

df['se_reduction_factor'] = df['standard_error_plain']/df['standard_error_control']

df['var_reduction_pct'] = (1 - (df["standard_error_control"] ** 2 / df["standard_error_plain"] ** 2))*100

df[["var_reduction_factor", "se_reduction_factor", "var_reduction_pct"]].describe()

In [ ]:
df.columns

In [ ]:
df.nlargest(20, 'relative_standard_error_control')[["L_over_V0", "S0_over_K", "sigma_s", "sigma_v", "mu_v", "rho", "CVA_control", "CVA_plain", "standard_error_plain", "standard_error_control", "relative_standard_error_plain", "relative_standard_error_control", "elapsed_time_seconds_plain", "elapsed_time_seconds_control", "var_reduction_factor", "se_reduction_factor", "var_reduction_pct"]]

In [ ]:
print("Mean variance reduction factor:", df["var_reduction_factor"].mean())

print("Median variance reduction factor:", df["var_reduction_factor"].median())

print("Mean SE reduction factor:", df["se_reduction_factor"].mean())

print("Median SE reduction factor:", df["se_reduction_factor"].median())

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["var_reduction_factor"], bins=40)

plt.xlabel("Variance Reduction Factor")
plt.ylabel("Count")
plt.title("Distribution of Variance Reduction")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["se_reduction_factor"], bins=40)

plt.xlabel("SE Reduction Factor")
plt.ylabel("Count")
plt.title("Standard Error Reduction")

plt.show()

In [ ]:
corr = np.corrcoef(
    df["CVA_plain"],
    df["CVA_control"]
)[0,1]

print("Correlation =", corr)

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(df["CVA_plain"], df["CVA_control"], alpha=0.4)

lims = [
    min(df["CVA_plain"].min(), df["CVA_control"].min()),
    max(df["CVA_plain"].max(), df["CVA_control"].max())
]

plt.plot(lims, lims, "r--")

plt.xlabel("Plain CVA")
plt.ylabel("Control CVA")

plt.title("Control vs Plain CVA")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["relative_standard_error_plain"], bins=40, alpha=0.5, label="Plain")

plt.hist(df["relative_standard_error_control"], bins=40, alpha=0.5, label="Control")

plt.legend()

plt.xlabel("Relative Standard Error")
plt.ylabel("Count")

plt.title("RSE Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(np.log10(df["CVA_control"]), df["relative_standard_error_control"], alpha=0.4)

plt.xlabel("log10(CVA)")
plt.ylabel("Relative Standard Error")

plt.title("RSE vs CVA")

plt.show()

In [ ]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20]

for t in thresholds:

    frac_plain = (df["relative_standard_error_plain"] > t).mean()

    frac_control = (df["relative_standard_error_control"] > t).mean()

    print(
        f"RSE > {100*t:.0f}% "
        f"| Plain: {100*frac_plain:.2f}% "
        f"| Control: {100*frac_control:.2f}%"
)

In [ ]:
bad = df["relative_standard_error_control"] > 0.05

print(df.loc[bad, "CVA_control"].describe())
print(df.loc[~bad, "CVA_control"].describe())

In [ ]:
se_thresholds = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

for t in se_thresholds:

    frac_plain = (df["standard_error_plain"] > t).mean()

    frac_control = (df["standard_error_control"] > t).mean()

    print(
        f"SE > {t:.0e}"
        f" | Plain: {100*frac_plain:.2f}%"
        f" | Control: {100*frac_control:.2f}%"
    )

In [ ]:
print(df['elapsed_time_seconds_control'].mean())
print(df['elapsed_time_seconds_plain'].mean())

In [ ]:
df.groupby(["S0_over_K", "sigma_v", "sigma_s", "rho"])["elapsed_time_seconds_control"].mean()

In [ ]:
params = ["S0_over_K", "sigma_s", "sigma_v", "rho"]

for param in params:
    stats = (
        df.groupby(param)["elapsed_time_seconds_plain"]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values(param)
    )

    plt.figure(figsize=(8,5))

    plt.plot(stats[param], stats["mean"], marker="o")

    plt.fill_between(
        stats[param],
        stats["mean"] - stats["std"],
        stats["mean"] + stats["std"],
        alpha=0.3
    )

    plt.xlabel(param)
    plt.ylabel("Elapsed time (s)")
    plt.title(f"Elapsed time vs {param}")
    plt.grid(True)
    plt.show()

In [ ]:
df["cva_ind"] = df.apply(lambda x: cva_call_option_ind(V0=100, r_v=x["mu_v"], sigma_v=x["sigma_v"], T=1, L=100*x["L_over_V0"], S0=100, r_s=0.02, sigma_s=x["sigma_s"], K=100/x["S0_over_K"], recovery_rate=0.4), axis=1)

In [ ]:
df.describe()

In [ ]:
df.head(20)

In [ ]:
corr = df['CVA'].corr(df['cva_ind'])
var_cva = df['CVA'].var()

In [ ]:
print(corr, var_cva)

In [ ]:
c = -corr / var_cva

In [ ]:
print(c)

In [ ]:
df_control = pd.read_csv("data/black_cox_cva_control_timing_results.csv")

In [ ]:
df_control.describe()

In [ ]:
df_control.head(20)

In [ ]:
df.nlargest(20, "relative_standard_error")

In [ ]:
df_control.nlargest(20, "relative_standard_error")

In [ ]:
df2 = pd.concat([df['CVA'], df_control['CVA']], axis=1)

In [ ]:
df2.head(30)

In [ ]:
df2.corr()

In [ ]:
df_control['relative_standard_error'].mean()

In [ ]:
df['relative_standard_error'].mean()

In [ ]:
df_control['standard_error'].mean()

In [ ]:
df['standard_error'].mean()

In [ ]:
df_mc = pd.read_csv("data/MC_parameters_tables.csv")

In [ ]:
df_mc